# PSI WUR PreProcessing (Notebook)

Notebook version of `PSI_WUR_PreProcessing.py`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bota-nick/WUR_Reflectance/blob/main/PSI_WUR_PreProcessing.ipynb)

In [ ]:
import sys

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    !pip install spectral
    print("Installed spectral library for Colab environment.")
    
    # 1. Clone the repository
    !git clone https://github.com/bota-nick/WUR_Reflectance.git

    # 2. Move into the repository directory
    %cd WUR_Reflectance

    # 3. List files to confirm you're in the right place
    !ls

In [ ]:
import spectral
import numpy as np
import os
import matplotlib.pyplot as plt
import pandas as pd
from glob import glob
import cv2

import zipfile

In [ ]:
def Bil2ENVI(path, output_path):

  #also read the header file as well to get dimensions, it will have the same name but .hdr extension
  hdr_path = os.path.splitext(path)[0] + '.hdr'
  if os.path.exists(hdr_path):
    with open(hdr_path, 'r') as hdr:
      lines = hdr.readlines()
      wavelengths = []
      reading_wavelengths = False
      for line in lines:
        if 'NCOLS' in line:
          cols = int(line.split()[-1])
        elif 'samples = ' in line:
          cols = int(line.split()[-1])
        elif 'NROWS' in line:
          rows = int(line.split()[-1])
        elif 'lines = ' in line:
          rows = int(line.split()[-1])
        elif 'NBANDS' in line:
          bands = int(line.split()[-1])
        elif 'bands = ' in line:
          bands = int(line.split()[-1])
        elif 'NBITS' in line:
          bits = int(line.split()[-1])
        elif 'data type = ' in line:
          datatype = int(line.split()[-1])
          if datatype == 4:
            bits = 32
          elif datatype == 2:
            bits = 16
        elif 'LAYOUT' in line:
          layout = line.split()[-1]
        elif 'interleave = ' in line:
          layout = line.split()[-1]
        elif 'WAVELENGTH' in line:
          reading_wavelengths = True
          continue
        elif 'WAVELENGTH END' in line:
          reading_wavelengths = False
          continue
        if reading_wavelengths:
          try:
            wavelengths.append(float(line.strip()))
          except ValueError:
            pass
        elif 'wavelength' in line.lower() and '=' in line and '{' in line and '}' in line:
          try:
            wl_str = line.split('=',1)[1]
            wl_str = wl_str.strip().lstrip('{').rstrip('}').replace('{','').replace('}','')
            wavelengths += [float(w.strip()) for w in wl_str.split(',') if w.strip()]
          except Exception:
            pass
  else:
    # If no header file, assume default dimensions (example: 512x512)
    rows = 512
    cols = 512
    bands = 100
    bits = 32  # Assuming float32
  

    # Read the binary file using bits to determine dtype
  if bits == 32:
    dtype = np.float32  
  elif bits == 16:
    dtype = np.uint16
  else:
    raise ValueError("Unsupported bit depth: {}".format(bits))
  
  # Switch to the .bil data file (same name, .bil extension)
  data_path = os.path.splitext(path)[0] + '.bil'
  # Load the binary data using np.fromfile for all layouts
  if layout == 'BIL':
    data = np.fromfile(data_path, dtype=dtype).reshape((rows, bands, cols)).transpose(0, 2, 1)
  elif layout == 'bil':
    data = np.fromfile(data_path, dtype=dtype).reshape((rows, bands, cols)).transpose(0, 2, 1)
  elif layout == 'BIP':
    data = np.fromfile(data_path, dtype=dtype).reshape((rows, cols, bands))
  elif layout == 'bip':
    data = np.fromfile(data_path, dtype=dtype).reshape((rows, cols, bands))
  else:  # Default to BSQ
    data = np.fromfile(data_path, dtype=dtype).reshape((bands, rows, cols)).transpose(1, 2, 0)
  
  #Create a new metadata dictionary for ENVI
  meta = {
    'description': 'Converted from BIL format',
    'samples': cols,
    'lines': rows,
    'bands': bands,
    'data type': 4 if bits == 32 else 2,  # ENVI data type codes
    'interleave': layout.lower(),
    'byte order': 0,
    'wavelength': wavelengths if 'wavelengths' in locals() else []
  }

  #now we resave the data in ENVI format with the same name as the input file but with .envi extension

  envi_path = os.path.splitext(output_path)[0] + '.hdr'
  # Save result as .hdr/.raw pair
  spectral.envi.save_image(envi_path, data, dtype=np.float32, force=True, metadata=meta)

        # Rename the .img file to .raw
  img_file = os.path.splitext(output_path)[0] + '.img'
  raw_file = os.path.splitext(output_path)[0]  + '.raw'
  if os.path.exists(img_file):
    os.rename(img_file, raw_file)

In [ ]:
bils = sorted(glob("data/*.bil")) 
for bil in bils:
  output_path = bil.replace("data/", "data/envi/").replace(".bil", ".hdr")
  os.makedirs(os.path.dirname(output_path), exist_ok=True)
  Bil2ENVI(bil, output_path)
  print(f"Converted {bil} to {output_path}")

In [ ]:
def Whiteref_Calibration(image_folder, white_ref_path, dark_ref_path):
    """
    Calibrate all images in image_folder using per-column mean (across rows) of white/dark references.
    Reflectance-like correction: (img - dark) / (white - dark).
    Saves corrected images to 'Corrected_ENVIs' subfolder.
    """
    import os
    import numpy as np
    import spectral
    from glob import glob

    zeroplus = 1e-10

    # Load white & dark reference
    white_ref_img = spectral.envi.open(white_ref_path)
    dark_ref_img  = spectral.envi.open(dark_ref_path)

    white_ref = white_ref_img.load().astype(np.float32)
    dark_ref  = dark_ref_img.load().astype(np.float32)

    if white_ref.ndim != 3 or dark_ref.ndim != 3:
        raise ValueError("White/Dark references must be 3D arrays: (rows, cols, bands).")

    # Mean across rows -> (cols, bands)
    white_col_mean = np.nanmedian(white_ref, axis=0)
    dark_col_mean  = np.nanmedian(dark_ref, axis=0)

    # Expand to (1, cols, bands) for safe broadcasting over rows
    white_cb = white_col_mean[np.newaxis, :, :]
    dark_cb  = dark_col_mean[np.newaxis, :, :]

    denom = (white_cb - dark_cb) + zeroplus

    corrected_dir = os.path.join(image_folder, 'Corrected_ENVIs')
    os.makedirs(corrected_dir, exist_ok=True)

    image_paths = glob(os.path.join(image_folder, '*.hdr'))

    for idx, hdr_path in enumerate(image_paths, 1):
        envi_img = spectral.envi.open(hdr_path)
        img = envi_img.load().astype(np.float32)
        meta = envi_img.metadata.copy()

        if img.ndim != 3:
            raise ValueError(f"{hdr_path} is not a 3D ENVI cube.")

        if img.shape[1:] != white_col_mean.shape:
            raise ValueError(
                f"Shape mismatch for {os.path.basename(hdr_path)}: "
                f"img cols/bands={img.shape[1:]} vs white/dark cols/bands={white_col_mean.shape}"
            )

        corrected = (img - dark_cb) / denom

        # Optional: clip reflectance range (uncomment if you want)
        # corrected = np.clip(corrected, 0, 2)

        out_hdr = os.path.join(corrected_dir, os.path.basename(hdr_path))
        spectral.envi.save_image(out_hdr, corrected, dtype=np.float32, force=True, metadata=meta)

        # Rename .img -> .raw (only if SPy produced .img)
        img_file = os.path.splitext(out_hdr)[0] + '.img'
        raw_file = os.path.splitext(out_hdr)[0] + '.raw'
        if os.path.exists(img_file) and not os.path.exists(raw_file):
            os.rename(img_file, raw_file)

        print(f"Corrected {idx}/{len(image_paths)} → {out_hdr}")


In [ ]:


# 1. Find the zip file in the data directory
zip_files = glob.glob('data/*.zip')

if zip_files:
    zip_path = zip_files[0]  # Take the first zip found
    target_dir = 'data'
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for member in zip_ref.infolist():
            # Skip directories themselves
            if member.is_dir():
                continue
            
            # Get just the filename, ignoring any internal zip folders
            filename = os.path.basename(member.filename)
            if not filename:
                continue
                
            # Construct the flat path
            source = zip_ref.open(member)
            target_path = os.path.join(target_dir, filename)
            
            # Write the file directly to /data
            with open(target_path, "wb") as target_file:
                target_file.write(source.read())
                
    print(f"Extracted all files from {zip_path} directly into {target_dir}/")
else:
    print("No zip file found in the data directory.")

In [ ]:
# 2) Calibrate all ENVIs in folder using the white calibration frame
Whiteref_Calibration(
    image_folder="data/envi",
    white_ref_path="data/envi/2025-12-16--13-51-40_round-0_cam-2_calibFrame.hdr",
    dark_ref_path="data/envi/2025-12-16--13-51-36_round-0_cam-2_calibFrame.hdr"
        
    
)

In [ ]:
def plant_mask(image_path, threshold=1.5):
    """
    Create a plant mask for the spectral using a water band index:
    - A: mean of bands with wavelength 120-130 (1200 nm - 1223 nm)
    - B: mean of bands with wavelength 210-220 (1423 nm -1448 nm)
    - Mask: (A/B) >= threshold
    Saves the mask as a PNG in a 'plant_masks' subfolder under the image's base folder.
    """
    img = spectral.envi.open(image_path)
    data = img.load().astype(np.float32)

    # Do the math to create the mask
    A = np.mean(data[..., 119:131], axis=-1)
    B = np.mean(data[..., 209:231], axis=-1)
    index = A / (B + 1e-10)
    mask = (index >= threshold).astype(np.uint8) * 255

    # Prepare output path
    base_folder = os.path.dirname(image_path)
    mask_folder = os.path.join(base_folder, 'plant_masks')
    os.makedirs(mask_folder, exist_ok=True)
    mask_filename = os.path.splitext(os.path.basename(image_path))[0] + '_mask.png'
    mask_path = os.path.join(mask_folder, mask_filename)
    cv2.imwrite(mask_path, mask)
    print(f"Saved plant mask to {mask_path}")


# 3) Optional mask on a corrected image (after calibration)
image_path = "data/envi/Corrected_ENVIs/2025-12-16--13-53-46_round-0_cam-2_tray-NPEC_G8_3x3_021.hdr"
plant_mask(
    image_path=image_path,
    threshold=1.5
)


In [ ]:
img = spectral.envi.open(image_path)
data = img.load().astype(np.float32)
data.shape

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def normalize(x):
    x_min, x_max = np.percentile(x, (1, 99))  # robust normalization
    return np.clip((x - x_min) / (x_max - x_min), 0, 1)

def plot_cube(arr, x, y, l):
    # Define band indices (Python is 0-based)
    r_band = 49                  # 50th band
    g_band = arr.shape[0] // 2   # middle band
    b_band = arr.shape[0] - 50   # last 50th band
    print(f"R={r_band}, G={g_band}, B={b_band}")

    # Extract and stack RGB
    rgb = np.stack(
        [arr[:,:,r_band], arr[:,:,g_band], arr[:,:,b_band]],
        axis=-1
    ).astype(np.float32)

    # Normalize each channel
    rgb_norm = np.stack(
        [normalize(rgb[..., i]) for i in range(3)],
        axis=-1
    )

    # Plot
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(rgb_norm)

    # Draw square (x, y is bottom-left corner)
    square = Rectangle(
        (x, y),      # bottom-left corner
        l,            # width
        l,            # height
        linewidth=2,
        edgecolor='red',
        facecolor='none'
    )
    ax.add_patch(square)

    # Optional: mark the bottom-left corner
    # ax.scatter(x, y, c='yellow', s=60, marker='x')

    ax.set_title("RGB Composite with Square ROI")
    # ax.axis("off")
    plt.show()
    return arr[y:y+l, x:x+l, :]

ref99 = plot_cube(data, 10, 30,10)
ref75 = plot_cube(data, 285, 50,10)
ref50 = plot_cube(data, 490, 50, 10)

In [ ]:
def calculate_mean(cube):
    return np.mean(cube, axis=(0, 1)) 


mean_ref99 = calculate_mean(ref99)
mean_ref75 = calculate_mean(ref75)
mean_ref50 = calculate_mean(ref50)

wavelengths = spectral.envi.open(image_path).bands.centers


plt.plot(wavelengths, mean_ref99, label='Spectralon 99%')
plt.plot(wavelengths, mean_ref75, label='Spectralon 75%')
plt.plot(wavelengths, mean_ref50, label='Spectralon 50%')
plt.xlabel("Wavelength (nm)")
plt.ylabel("Reflectance")
plt.xlim(1000, 1500)
plt.legend()

In [ ]:
# Calculate the normalised reflectance
normalised_ref75 = (mean_ref75 - mean_ref50) / (mean_ref99 - mean_ref50 + 1e-10)

# Create the plot
plt.plot(wavelengths, normalised_ref75, label='Normalised Spectralon 75%')

# Add a horizontal line at y = 0.5
plt.axhline(y=0.5, color='r', linestyle='--', label='0.5 Threshold')

# Set the x-axis range from 1000 to 1500
plt.xlim(1000, 1500)
plt.ylim(0.3, 0.7)

# Add labels and legend
plt.xlabel("Wavelength (nm)")
plt.ylabel("Reflectance")
plt.legend()

# Save or display the plot
plt.show()